# **Regression: K-Nearest Neighbors Regressor (K-NN)**

## **Justification of Preprocessing Strategy**

### **Distance-Based Model Requirements (Feature Scaling)**
K-NN Regressor is a non-parametric, distance-based algorithm. To predict the `diabetes_risk_score` for a new patient, it identifies the $k$ closest training examples in the feature space and calculates the average of their risk scores. Because it strictly relies on geometric distances (Euclidean or Manhattan), features with naturally large numerical ranges would completely dominate features with smaller scales. Therefore, feature scaling is absolutely mandatory. We will evaluate both **Standardization** and **Normalization** to determine which continuous space yields the most accurate spatial averages.

### **The Overfitting Trap in Regression (`weights='distance'`)**
Just like in classification, K-NN regression is highly susceptible to extreme overfitting. If we set $k=1$, or if we use `weights='distance'`, the model will assign infinite weight to a training point's distance to itself. This results in a perfect memorization of the training set (Training RMSE = 0.0), but poor generalization to new data. To monitor this, we log **both Train and Test metrics (RMSE, MAE, R²)** simultaneously.

## **Experiment Design**

We designed a tournament of 6 experiments (2 Scalers × 3 Optimization Levels) to identify the most accurate and computationally efficient configuration:

* **Baseline**: Scikit-Learn defaults ($k=5$, uniform weights, Euclidean distance) to establish a performance floor in both scaled spaces.
* **GridSearchCV**: An exhaustive, 5-fold cross-validated search exploring discrete steps of $k$ (safely avoiding 1), weight formulas, and distance metrics ($p=1$ vs $p=2$).
* **Optuna Optimization**: Bayesian search to dynamically explore a continuous parameter space for $k$, targeting the minimization of the Root Mean Squared Error (RMSE) on the validation folds.

In [2]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_KNN")

# 2. Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

# Categorical columns for One-Hot Encoding
categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features and target
# Note: Dropping the classification targets to prevent leakage!
X = df_final.drop(["diagnosed_diabetes", "diabetes_stage", "diabetes_risk_score"], axis=1)
y = df_final['diabetes_risk_score']

# Split data (80/20) - No stratify needed for continuous regression targets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify numerical columns for scaling
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

SEED = 42

def log_regression_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to explicitly monitor the Overfitting Gap"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Metrics
    mlflow.log_metric("rmse_train", mean_squared_error(y_tr, y_tr_pred) ** 0.5)
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr, y_tr_pred))
    mlflow.log_metric("r2_train", r2_score(y_tr, y_tr_pred))
    
    # Test Metrics
    mlflow.log_metric("rmse_test", mean_squared_error(y_te, y_te_pred) ** 0.5)
    mlflow.log_metric("mae_test", mean_absolute_error(y_te, y_te_pred))
    mlflow.log_metric("r2_test", r2_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# 6 RUNS TOURNAMENT (2 Scalers x 3 Optimization Levels)
# ---------------------------------------------------------
scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

for s_name, scaler in scalers.items():
    # Apply Feature Scaling
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

    # --- RUN 1 & 4: BASELINE ---
    with mlflow.start_run(run_name=f"KNN_Reg_{s_name}_Baseline"):
        # Defaults: n_neighbors=5, weights='uniform', p=2
        model = KNeighborsRegressor(n_jobs=-1)
        
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        mlflow.log_params(model.get_params())
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "none_default")
        
        log_regression_metrics(model, X_train_scaled, y_train, X_test_scaled, y_test, duration)

    # --- RUN 2 & 5: GRIDSEARCHCV ---
    with mlflow.start_run(run_name=f"KNN_Reg_{s_name}_GridSearch"):
        # Avoiding k=1 to force generalization
        param_grid = {
            'n_neighbors': [5, 15, 25, 35],
            'weights': ['uniform', 'distance'],
            'p': [1, 2] # 1: Manhattan, 2: Euclidean
        }
        
        grid = GridSearchCV(
            KNeighborsRegressor(n_jobs=-1), 
            param_grid, 
            cv=KFold(n_splits=5, shuffle=True, random_state=SEED), 
            scoring='neg_root_mean_squared_error', 
            n_jobs=-1
        )
        
        start_time = time.time()
        grid.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        best_knn_grid = grid.best_estimator_
        
        mlflow.log_params(grid.best_params_)
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "GridSearchCV")
        
        log_regression_metrics(best_knn_grid, X_train_scaled, y_train, X_test_scaled, y_test, duration)

    # --- RUN 3 & 6: OPTUNA ---
    def objective(trial):
        params = {
            "n_neighbors": trial.suggest_int("n_neighbors", 5, 50),
            "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
            "p": trial.suggest_int("p", 1, 2)
        }
        
        knn_opt = KNeighborsRegressor(**params, n_jobs=-1)
        
        # Internal Cross-Validation on Train Data Only
        scores = cross_val_score(
            knn_opt, X_train_scaled, y_train, 
            cv=KFold(n_splits=3, shuffle=True, random_state=SEED), 
            scoring='neg_root_mean_squared_error', 
            n_jobs=-1
        )
        # Optuna minimizes by default, but cross_val_score returns negative RMSE. 
        # We return the positive RMSE to minimize it.
        return -scores.mean()

    with mlflow.start_run(run_name=f"KNN_Reg_{s_name}_Optuna"):
        # Direction is minimize because we are tracking RMSE
        study = optuna.create_study(direction="minimize")
        start_time = time.time()
        study.optimize(objective, n_trials=15) 
        duration = time.time() - start_time
        
        # Retrain best found model
        best_knn_opt = KNeighborsRegressor(**study.best_params, n_jobs=-1)
        best_knn_opt.fit(X_train_scaled, y_train)
        
        mlflow.log_params(study.best_params)
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "optuna")
        
        log_regression_metrics(best_knn_opt, X_train_scaled, y_train, X_test_scaled, y_test, duration)

[I 2026-05-22 11:04:02,067] A new study created in memory with name: no-name-f58de76a-0a81-4be6-bb48-6f9bf7ff4f5d
[I 2026-05-22 11:04:49,109] Trial 0 finished with value: 2.927253220235519 and parameters: {'n_neighbors': 35, 'weights': 'uniform', 'p': 1}. Best is trial 0 with value: 2.927253220235519.
[I 2026-05-22 11:04:57,720] Trial 1 finished with value: 2.6845782888425553 and parameters: {'n_neighbors': 46, 'weights': 'distance', 'p': 2}. Best is trial 1 with value: 2.6845782888425553.
[I 2026-05-22 11:05:05,804] Trial 2 finished with value: 2.6886612502200573 and parameters: {'n_neighbors': 47, 'weights': 'distance', 'p': 2}. Best is trial 1 with value: 2.6845782888425553.
[I 2026-05-22 11:05:13,704] Trial 3 finished with value: 2.5471486340452363 and parameters: {'n_neighbors': 13, 'weights': 'distance', 'p': 2}. Best is trial 3 with value: 2.5471486340452363.
[I 2026-05-22 11:05:21,686] Trial 4 finished with value: 2.6388604519423744 and parameters: {'n_neighbors': 33, 'weights'

# Winner Run Selection (Priority Elimination Framework)

### Policy
A run is only eligible to win if it does NOT show evidence of overfitting or underfitting. Before applying the MAE/RMSE/R² decision rules, we require the Train→Test gap to remain small enough to indicate acceptable generalization. Runs that memorize the training set or show a large Train/Test gap are disqualified regardless of metric rank.

### Selection Criteria (priority order)
1. **Generalization filter (mandatory):** runs with overfitting or underfitting are removed from consideration.
2. **Priority 1 (60%): Lowest MAE (Test)** — primary objective for regression accuracy.
3. **Priority 2 (30%): Lowest RMSE (Test)** — used to reject runs where RMSE grows disproportionately relative to MAE.
4. **Priority 3 (10%): Acceptable R² (Test)** — used as a quality check to confirm the model explains the target variance well enough.
5. **Tiebreaker: Lowest Fit Time** — if MAE, RMSE, and R² are effectively tied.

### Runs Summary

| Run | Scaler | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| KNN_Reg_Standardization_Baseline | Standardization | 1.67987 | 2.06872 | 2.12317 | 2.60963 | 0.94503 | 0.91748 | 0.04s |
| KNN_Reg_Normalization_Baseline | Normalization | 2.57858 | 3.20603 | 3.26145 | 4.03081 | 0.87029 | 0.80313 | 0.04s |
| KNN_Reg_Standardization_GridSearch | Standardization | 0.00000 | 1.96202 | 0.00000 | 2.47414 | 1.00000 | 0.92583 | 206.18s |
| KNN_Reg_Standardization_Optuna | Standardization | 0.00000 | 1.96239 | 0.00000 | 2.47337 | 1.00000 | 0.92587 | 314.50s |
| KNN_Reg_Normalization_GridSearch | Normalization | 0.00000 | 2.86072 | 0.00000 | 3.59528 | 1.00000 | 0.84337 | 199.41s |
| KNN_Reg_Normalization_Optuna | Normalization | 0.00000 | 2.88110 | 0.00000 | 3.61341 | 1.00000 | 0.84179 | 417.62s |

### Generalization Check
- **KNN_Reg_Standardization_Baseline:** Train and test metrics stay reasonably close, so it passes the generalization filter.
- **KNN_Reg_Normalization_Baseline:** Passes the filter, but its test errors are clearly worse than the standardization baseline.
- **KNN_Reg_Standardization_GridSearch:** Disqualified (overfitting: Train MAE = 0 and Train RMSE = 0).
- **KNN_Reg_Standardization_Optuna:** Disqualified (overfitting: Train MAE = 0 and Train RMSE = 0).
- **KNN_Reg_Normalization_GridSearch:** Disqualified (overfitting: Train MAE = 0 and Train RMSE = 0).
- **KNN_Reg_Normalization_Optuna:** Disqualified (overfitting: Train MAE = 0 and Train RMSE = 0).

### Step-by-Step Elimination
**Step 1 — Apply the generalization filter**
- Passing runs: KNN_Reg_Standardization_Baseline, KNN_Reg_Normalization_Baseline.

**Step 2 — Compare Test MAE (Priority 1 — 60%)**
- KNN_Reg_Standardization_Baseline: 2.06872
- KNN_Reg_Normalization_Baseline: 3.20603
- Lowest MAE: **KNN_Reg_Standardization_Baseline**.

**Step 3 — Compare Test RMSE (Priority 2 — 30%)**
- KNN_Reg_Standardization_Baseline: 2.60963
- KNN_Reg_Normalization_Baseline: 4.03081
- The winner also has the lower RMSE, so no rejection is needed.

**Step 4 — Check Test R² (Priority 3 — 10%)**
- KNN_Reg_Standardization_Baseline: 0.91748
- KNN_Reg_Normalization_Baseline: 0.80313
- Both are acceptable, but the standardization baseline is clearly stronger.

### Final Decision
**Winner: KNN_Reg_Standardization_Baseline**

**Justification:** Among the runs that are not disqualified for overfitting, `KNN_Reg_Standardization_Baseline` has the lowest Test MAE, the lowest Test RMSE, and a strong Test R². Fit time is not needed as a tiebreaker.

## Winner Hyperparameters
| Parameter | Value |
|---|---|
| **Scaler** | Standardization |
| **n_neighbors** | 5 |
| **weights** | uniform |
| **p** | 2 |
| **n_jobs** | -1 |

## Overfitting / Underfitting Diagnosis
- The GridSearch and Optuna runs are disqualified because they memorize the training set almost perfectly (Train MAE = 0 and Train RMSE = 0), which is a clear sign of overfitting.
- `KNN_Reg_Standardization_Baseline` generalizes better than the other eligible run and is therefore the best operational choice.

**Operational note:** This selection is based on manual analysis of the logged metrics, not on scripts.